# Target encoding
In this notebook I will explore the Target encoding method for every non-numerical feature. More specifically,

`"age"`: stays as it is

`"workclass"`: Target encoding

`"fnlwgt"`: drop

`"education"`: Target encoding

`"education-num"`: drop as `"education"` is represented by Target encoding

`"marital-status"`: Target encoding

`"occupation"`: Target encoding

`"relationship"`: Target encoding

`"race"`: Target encoding

`"sex"`: Target encoding

`"capital-gain"`: stays as it is

`"capital-loss"`: stays as it is

`"hours-per-week"`: stays as it is

`"native-country"`: Target encoding


Here is the raw data loaded and apply
$$
y =
\begin{cases}
1 & \text{if income } > \$50K \\
0 & \text{otherwise}
\end{cases}
$$
to the data set. Also drop the columns which were stated before 

In [1]:
# Enable autoreloading of imported modules
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import pandas as pd
from courselib.utils.splits import train_test_split

# Add the repo root (two levels up from this notebook) to sys.path
sys.path.insert(0, os.path.abspath("../../"))

In [2]:
file_name = "adult.data"
column_names = column_names = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "Class"
]
df = pd.read_csv(file_name, names= column_names,skipinitialspace=True)

In [ ]:
target_variable_map = {
    "<=50K": 0,
    ">50K": 1,
}

columns_to_be_dropped = [
    "fnlwgt"
]

In [4]:
df= df.drop(columns=columns_to_be_dropped)
df["Class"] = df["Class"].map(target_variable_map)
df

,age,workclass,education,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,Class
0,39,State-gov,Bachelors,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,0
1,50,Self-emp-not-inc,Bachelors,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,0
2,38,Private,HS-grad,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0
3,53,Private,11th,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0
4,28,Private,Bachelors,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,Assoc-acdm,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,0
32557,40,Private,HS-grad,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,1
32558,58,Private,HS-grad,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,0
32559,22,Private,HS-grad,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,0


## One-Hot-Encoding method

In [5]:
def Target_Encoder(df,column_to_be_encoded,target_column_name, delete_old_column = False, drop_unknown_data_rows = [] ):
    """takes a pandas dataframe and columns which need to be encoded and creates new column called "{oldcolumnname}_target".
       The new column has then the target representation
       outputs then a new edited copy of df.

    Args:
        df (pd.dataframe): just the dataset
        column_to_be_encoded (list): a list of column names form df which need to be encoded
        target_column_name (str): the column name of the target variable
        delete_old_column (bool, optional): Wether the columns form column_to_be_encoded are deleted or not. Defaults to False.
        drop_unknown_data_rows (list, optional): Drops all rows which include the strings which are in this list. Defaults to empty list.
    """
    _df = df.copy()
    row_amount_before = len(_df)

    if type(column_to_be_encoded) == str:
        column_to_be_encoded = [column_to_be_encoded]

    if drop_unknown_data_rows !=[]:
        for column_name in column_to_be_encoded:
            _df = _df[~_df[column_name].isin(drop_unknown_data_rows)]

    amount_of_rows_deleted = row_amount_before - len(_df)

    _df = _df.reset_index(drop=True)
    for i,column_name in enumerate(column_to_be_encoded):
        
        new_column_name = f"{column_name}_target"
        counting_dataframe = _df.groupby(column_name)[target_column_name].mean().to_frame().T.reset_index(drop=True) # gives me every catgory and their respective target mean with col names of the categories and the 0-th line their respective target mean
        
        _df[new_column_name] = _df[column_name].map(counting_dataframe.loc[0])
        
    if delete_old_column:
        _df = _df.drop(columns=column_to_be_encoded)
        
    print(f"{amount_of_rows_deleted} out of {row_amount_before} were deleted, ie. {(1- amount_of_rows_deleted/row_amount_before)*100}% still remain ")            
    return _df

## Formatting the Data Set

### Choose the Columns which need to be encoded

In [6]:
target_encoding_cols = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",
]


### Running the Data though the One-Hot-Encoder

In [7]:
from projectlib.encoding import drop_rows

df_encoded = drop_rows(df,["?"])
df_encoded = Target_Encoder(df,target_encoding_cols,"Class",delete_old_column=True)

2399 out of 32561 were deleted, ie.92.63229016307852% still remain 
0 out of 32561 were deleted, ie. 100.0% still remain 


In [8]:
df_encoded

,age,capital-gain,capital-loss,hours-per-week,Class,workclass_target,education_target,marital-status_target,occupation_target,relationship_target,race_target,sex_target,native-country_target
0,39,2174,0,40,0,0.271957,0.414753,0.045961,0.134483,0.103070,0.25586,0.305737,0.245835
1,50,0,0,13,0,0.284927,0.414753,0.446848,0.484014,0.448571,0.25586,0.305737,0.245835
2,38,0,0,40,0,0.218673,0.159509,0.104209,0.062774,0.103070,0.25586,0.305737,0.245835
3,53,0,0,40,0,0.218673,0.051064,0.446848,0.062774,0.448571,0.12388,0.305737,0.245835
4,28,0,0,40,0,0.218673,0.414753,0.446848,0.449034,0.475128,0.12388,0.109461,0.263158
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,0,0,38,0,0.218673,0.248360,0.446848,0.304957,0.475128,0.25586,0.109461,0.245835
32557,40,0,0,40,1,0.218673,0.159509,0.446848,0.124875,0.448571,0.25586,0.305737,0.245835
32558,58,0,0,40,0,0.218673,0.159509,0.085599,0.134483,0.063262,0.25586,0.109461,0.245835
32559,22,0,0,20,0,0.218673,0.159509,0.045961,0.134483,0.013220,0.25586,0.305737,0.245835


### Test if the Encoded data Runs through the `train_test_split`

In [9]:
X, Y, X_train, Y_train, X_test, Y_test =  train_test_split(df_encoded, 
                                                           training_data_fraction=0.8, 
                                                           return_numpy=True)

print('Training data split as follows:')
print(f'  Training data samples: {len(X_train)}')
print(f'      Test data samples: {len(X_test)}')

Training data split as follows:
  Training data samples: 26049
      Test data samples: 6512
